# Results Analysis

Per-dataset tables and win-rate summaries built from the committed reports on this branch
via the helpers in `bin/`.

- **Single-model** results come from `exp/tuned/` (fixed tuning budget).
- **Ensemble** results come from `exp/ensembled/` (greedy Caruana; 100 members / 20 for
  tabred & microsoft). Only the MLP, TabM and RealMLP families have ensembles.
- **Main tables**: rows = methods, columns = datasets.
- **Win rates**: agentic HP set vs the baseline HP set, per model family.

In [1]:
from IPython.display import display
import numpy as np
import pandas as pd

from bin.analysis.report_io import load_single_results
from bin.analysis.ensemble_io import load_committed_ensemble_results
from bin.analysis.metrics import pct_improvement_from_errors, canonical_metric_name
from bin.analysis.results_config import (
    MAX_BUDGET, TABRED_BUDGET, DATASETS, SINGLE_METHODS,
    METHOD_DISPLAY, FAMILY_PAIRS, FAMILY_DISPLAY, ENSEMBLE_TO_MAIN_METHOD,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)

print(f"Budget={MAX_BUDGET}; TabRed and microsoft use budget {TABRED_BUDGET}; "
      f"TabICL uses its full grid report.")

# Single-model results (exp/tuned/).
results = load_single_results(budget=MAX_BUDGET)

# Greedy-ensemble results (exp/ensembled/), mapped to their main method.
ensemble = load_committed_ensemble_results()
ensemble["main_method"] = ensemble["output_method"].where(
    ensemble["output_method"].notna(),
    ensemble["method"].map(ENSEMBLE_TO_MAIN_METHOD),
)
ensemble["metric"] = ensemble["score_name"].map(canonical_metric_name)

# dataset -> task type (reg/clf), taken from the single-model results.
TASK_BY_DATASET = results.drop_duplicates("dataset").set_index("dataset")["task_type"].to_dict()
# Ensemble methods follow SINGLE_METHODS order, then any ensemble-only methods (e.g. v4).
ens_present = list(dict.fromkeys(ensemble["main_method"].dropna()))
ENS_METHODS = [m for m in SINGLE_METHODS if m in ens_present] + [m for m in ens_present if m not in SINGLE_METHODS]
COVERAGE_METHODS = list(SINGLE_METHODS) + [m for m in ens_present if m not in SINGLE_METHODS]

print(f"Single: {len(results)} rows, {results['dataset'].nunique()} datasets, "
      f"{results['method'].nunique()} methods.")
print(f"Ensemble: {len(ensemble)} rows, {ensemble['dataset'].nunique()} datasets, "
      f"{len(ENS_METHODS)} methods.")

coverage = pd.DataFrame({
    "single": results.groupby("method")["dataset"].nunique(),
    "ensemble": ensemble.groupby("main_method")["dataset"].nunique(),
}).reindex(COVERAGE_METHODS)
coverage.insert(0, "method_display", coverage.index.map(METHOD_DISPLAY))
display(coverage)

Budget=200; TabRed and microsoft use budget 100; TabICL uses its full grid report.
Single: 450 rows, 45 datasets, 10 methods.
Ensemble: 360 rows, 45 datasets, 8 methods.


,method_display,single,ensemble
realmlp,RealMLP,45,45.0
agentic-realmlp-v8,RealMLP (agentic),45,45.0
mlp,MLP,45,45.0
agentic-mlp,MLP (agentic),45,45.0
lightgbm,LightGBM,45,45.0
agentic-lightgbm,LightGBM (agentic),45,45.0
tabm,TabM,45,45.0
agentic-tabm,TabM (agentic),45,45.0
tabicl,TabICL,45,NaN
agentic-tabicl,TabICL (agentic),45,NaN


In [2]:
# Shared helpers reused for the single-model and ensemble tables.

def fmt(mean, std=float("nan")):
    if pd.isna(mean):
        return ""
    return f"{mean:.4f}" if pd.isna(std) else f"{mean:.4f} +/- {std:.4f}"


def main_table(frame, methods, with_std):
    # rows = methods, columns = (dataset, metric); values = formatted mean [+/- std].
    present = [d for d in DATASETS if d in set(frame["dataset"])]
    metric_by_dataset = frame.drop_duplicates("dataset").set_index("dataset")["metric"]
    columns = pd.MultiIndex.from_tuples(
        [(d, metric_by_dataset[d]) for d in present], names=["dataset", "metric"],
    )
    if with_std:
        values = [fmt(m, s) for m, s in zip(frame["score"], frame["score_std"])]
    else:
        values = [fmt(m) for m in frame["score"]]
    disp = frame.assign(method_display=frame["method_key"].map(METHOD_DISPLAY), value=values)
    table = (
        disp.pivot_table(index="method_display", columns=["dataset", "metric"],
                         values="value", aggfunc="first")
        .reindex(index=[METHOD_DISPLAY[m] for m in methods], columns=columns)
        .fillna("")
    )
    table.index.name = "method"
    return table


def error_pivot(frame):
    return frame.pivot_table(index="dataset", columns="method_key", values="error", aggfunc="first")


def comparison_details(err, pairs=FAMILY_PAIRS):
    # Per-(family, dataset) improvement of the agentic HP set over the baseline HP set.
    records = []
    for family, agentic, base in pairs:  # (family, candidate=agentic, baseline=base)
        if agentic not in err.columns or base not in err.columns:
            continue
        sub = err[[agentic, base]].dropna()  # datasets where both were run
        for ds in sub.index:
            a, b = float(sub.at[ds, agentic]), float(sub.at[ds, base])
            records.append({
                "family": FAMILY_DISPLAY.get(family, family),
                "candidate": METHOD_DISPLAY.get(agentic, agentic),
                "baseline": METHOD_DISPLAY.get(base, base),
                "dataset": ds,
                "task_type": TASK_BY_DATASET.get(ds),
                "agentic_error": a,
                "baseline_error": b,
                "improvement_pct": pct_improvement_from_errors(b, a),
                "win": bool(a < b),
            })
    return pd.DataFrame(records)


def summarize(details, group_cols):
    if details.empty:
        return pd.DataFrame()
    return (
        details.groupby(group_cols, dropna=False, observed=True)
        .agg(
            datasets=("dataset", "nunique"),
            wins=("win", "sum"),
            win_rate_pct=("win", lambda v: round(float(v.mean()) * 100.0, 1)),
            mean_improvement_pct=("improvement_pct", lambda v: round(float(np.nanmean(v)), 2)),
            median_improvement_pct=("improvement_pct", lambda v: round(float(np.nanmedian(v)), 2)),
        )
        .reset_index()
    )


# Normalised frames (method_key / dataset / metric / score / score_std / error).
single_frame = results.rename(columns={"method": "method_key"})[
    ["method_key", "dataset", "metric", "score", "score_std", "error"]
]
ens_frame = ensemble.assign(method_key=ensemble["main_method"], score_std=np.nan)[
    ["method_key", "dataset", "metric", "score", "score_std", "error"]
]

## Single-model tables

Main table: rows = methods, columns = datasets. Values are the raw test metric
`mean +/- std` (metric shown in each column header). Blank = method not run on that dataset.

In [3]:
main_table(single_frame, SINGLE_METHODS, with_std=True)

dataset,black-friday,california,churn,diamond,house,microsoft,tabarena/APSFailure,tabarena/Amazon_employee_access,tabarena/Another-Dataset-on-used-Fiat-500,tabarena/Bioresponse,tabarena/Diabetes130US,tabarena/E-CommereShippingData,tabarena/Food_Delivery_Time,tabarena/GiveMeSomeCredit,tabarena/HR_Analytics_Job_Change_of_Data_Scientists,tabarena/NATICUSdroid,tabarena/QSAR-TID-11,tabarena/QSAR_fish_toxicity,tabarena/airfoil_self_noise,tabarena/bank-marketing,tabarena/concrete_compressive_strength,tabarena/credit_card_clients_default,tabarena/customer_satisfaction_in_airline,tabarena/diabetes,tabarena/healthcare_insurance_expenses,tabarena/heloc,tabarena/in_vehicle_coupon_recommendation,tabarena/jm1,tabarena/kddcup09_appetency,tabarena/miami_housing,tabarena/online_shoppers_intention,tabarena/physiochemical_protein,tabarena/polish_companies_bankruptcy,tabarena/qsar-biodeg,tabarena/splice,tabarena/taiwanese_bankruptcy_prediction,tabarena/wine_quality,tabred/cooking-time,tabred/delivery-eta,tabred/ecom-offers,tabred/homecredit-default,tabred/homesite-insurance,tabred/maps-routing,tabred/sberbank-housing,tabred/weather
metric,rmse,rmse,accuracy,rmse,rmse,rmse,roc_auc,roc_auc,rmse,roc_auc,roc_auc,roc_auc,rmse,roc_auc,roc_auc,roc_auc,rmse,rmse,rmse,roc_auc,rmse,roc_auc,roc_auc,roc_auc,rmse,roc_auc,roc_auc,roc_auc,roc_auc,rmse,roc_auc,rmse,roc_auc,roc_auc,log_loss,roc_auc,rmse,rmse,rmse,roc_auc,roc_auc,roc_auc,rmse,rmse,rmse
method,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
RealMLP,3449.6726 +/- 1.8948,0.4062 +/- 0.0022,0.8622 +/- 0.0013,526.7907 +/- 1.9909,30864.4004 +/- 286.0294,0.7418 +/- 0.0003,0.9931 +/- 0.0010,0.8448 +/- 0.0054,747.5714 +/- 11.9138,0.8696 +/- 0.0042,0.6771 +/- 0.0031,0.7413 +/- 0.0040,7.4429 +/- 0.0144,0.8701 +/- 0.0004,0.8109 +/- 0.0016,0.9844 +/- 0.0005,0.8480 +/- 0.0072,0.9233 +/- 0.0078,1.1872 +/- 0.0386,0.7669 +/- 0.0015,4.6979 +/- 0.0916,0.7905 +/- 0.0011,0.9951 +/- 0.0000,0.8002 +/- 0.0201,4261.7941 +/- 44.0584,0.7977 +/- 0.0002,0.8442 +/- 0.0015,0.7437 +/- 0.0040,0.8176 +/- 0.0028,84008.5048 +/- 1723.6740,0.9298 +/- 0.0014,3.2988 +/- 0.0134,0.9307 +/- 0.0090,0.9127 +/- 0.0050,0.1027 +/- 0.0046,0.9239 +/- 0.0175,0.6053 +/- 0.0030,0.4813 +/- 0.0006,0.5476 +/- 0.0010,0.5935 +/- 0.0020,0.8630 +/- 0.0013,0.9651 +/- 0.0003,0.1612 +/- 0.0002,0.2310 +/- 0.0009,1.4363 +/- 0.0020
RealMLP (agentic),3436.8781 +/- 1.0140,0.4071 +/- 0.0018,0.8622 +/- 0.0017,518.8477 +/- 1.9976,30734.1317 +/- 302.4184,0.7435 +/- 0.0002,0.9914 +/- 0.0014,0.8563 +/- 0.0063,757.9338 +/- 12.2266,0.8624 +/- 0.0037,0.6530 +/- 0.0340,0.7409 +/- 0.0036,7.4876 +/- 0.0108,0.8689 +/- 0.0004,0.8103 +/- 0.0022,0.9846 +/- 0.0005,0.8539 +/- 0.0037,0.9454 +/- 0.0156,1.1341 +/- 0.0186,0.7675 +/- 0.0020,4.6595 +/- 0.0599,0.7805 +/- 0.0053,0.9952 +/- 0.0001,0.8251 +/- 0.0114,4188.8642 +/- 54.5268,0.7942 +/- 0.0018,0.8419 +/- 0.0020,0.7416 +/- 0.0027,0.7674 +/- 0.0538,83701.4154 +/- 670.6805,0.9284 +/- 0.0008,3.2711 +/- 0.0112,0.9543 +/- 0.0042,0.9273 +/- 0.0026,0.1082 +/- 0.0500,0.9269 +/- 0.0258,0.6089 +/- 0.0028,0.4804 +/- 0.0003,0.5471 +/- 0.0010,0.5812 +/- 0.0174,0.8626 +/- 0.0005,0.9633 +/- 0.0005,0.1609 +/- 0.0002,0.2358 +/- 0.0023,1.4389 +/- 0.0016
MLP,3472.8162 +/- 2.4946,0.4535 +/- 0.0020,0.8611 +/- 0.0027,525.7104 +/- 2.7992,30996.8740 +/- 384.8644,0.7465 +/- 0.0004,0.9922 +/- 0.0006,0.8330 +/- 0.0015,738.2305 +/- 9.5736,0.8530 +/- 0.0049,0.6663 +/- 0.0010,0.7406 +/- 0.0016,7.6184 +/- 0.0065,0.8702 +/- 0.0002,0.8091 +/- 0.0011,0.9851 +/- 0.0003,0.8885 +/- 0.0062,0.9893 +/- 0.0344,1.2111 +/- 0.0851,0.7726 +/- 0.0005,4.8574 +/- 0.1416,0.7849 +/- 0.0028,0.9941 +/- 0.0001,0.8110 +/- 0.0080,4315.3953 +/- 61.0633,0.7953 +/- 0.0007,0.8245 +/- 0.0031,0.7311 +/- 0.0043,0.8146 +/- 0.0036,88228.9871 +/- 985.1957,0.9245 +/- 0.0015,3.5443 +/- 0.0155,0.9228 +/- 0.0060,0.9141 +/- 0.0037,0.1149 +/- 0.0019,0.9366 +/- 0.0007,0.6746 +/- 0.0043,0.4808 +/- 0.0004,0.5529 +/- 0.0016,0.5967 +/- 0.0018,0.8587 +/- 0.0014,0.9599 +/- 0.0008,0.1618 +/- 0.0002,0.2406 +/- 0.

### Metric-error view (lower is better everywhere)

In [4]:
(single_frame.pivot_table(index="method_key", columns="dataset", values="error", aggfunc="first")
 .reindex(index=SINGLE_METHODS, columns=[d for d in DATASETS if d in set(single_frame["dataset"])])
 .rename(index=METHOD_DISPLAY)
 .round(4))

dataset,black-friday,california,churn,diamond,house,microsoft,tabarena/APSFailure,tabarena/Amazon_employee_access,tabarena/Another-Dataset-on-used-Fiat-500,tabarena/Bioresponse,tabarena/Diabetes130US,tabarena/E-CommereShippingData,tabarena/Food_Delivery_Time,tabarena/GiveMeSomeCredit,tabarena/HR_Analytics_Job_Change_of_Data_Scientists,tabarena/NATICUSdroid,tabarena/QSAR-TID-11,tabarena/QSAR_fish_toxicity,tabarena/airfoil_self_noise,tabarena/bank-marketing,tabarena/concrete_compressive_strength,tabarena/credit_card_clients_default,tabarena/customer_satisfaction_in_airline,tabarena/diabetes,tabarena/healthcare_insurance_expenses,tabarena/heloc,tabarena/in_vehicle_coupon_recommendation,tabarena/jm1,tabarena/kddcup09_appetency,tabarena/miami_housing,tabarena/online_shoppers_intention,tabarena/physiochemical_protein,tabarena/polish_companies_bankruptcy,tabarena/qsar-biodeg,tabarena/splice,tabarena/taiwanese_bankruptcy_prediction,tabarena/wine_quality,tabred/cooking-time,tabred/delivery-eta,tabred/ecom-offers,tabred/homecredit-default,tabred/homesite-insurance,tabred/maps-routing,tabred/sberbank-housing,tabred/weather
method_key,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
RealMLP,3449.6726,0.4062,0.1378,526.7907,30864.4004,0.7418,0.0069,0.1552,747.5714,0.1304,0.3229,0.2587,7.4429,0.1299,0.1891,0.0156,0.8480,0.9233,1.1872,0.2331,4.6979,0.2095,0.0049,0.1998,4261.7941,0.2023,0.1558,0.2563,0.1824,84008.5048,0.0702,3.2988,0.0693,0.0873,0.1027,0.0761,0.6053,0.4813,0.5476,0.4065,0.1370,0.0349,0.1612,0.2310,1.4363
RealMLP (agentic),3436.8781,0.4071,0.1378,518.8477,30734.1317,0.7435,0.0086,0.1437,757.9338,0.1376,0.3470,0.2591,7.4876,0.1311,0.1897,0.0154,0.8539,0.9454,1.1341,0.2325,4.6595,0.2195,0.0048,0.1749,4188.8642,0.2058,0.1581,0.2584,0.2326,83701.4154,0.0716,3.2711,0.0457,0.0727,0.1082,0.0731,0.6089,0.4804,0.5471,0.4188,0.1374,0.0367,0.1609,0.2358,1.4389
MLP,3472.8162,0.4535,0.1389,525.7104,30996.8740,0.7465,0.0078,0.1670,738.2305,0.1470,0.3337,0.2594,7.6184,0.1298,0.1909,0.0149,0.8885,0.9893,1.2111,0.2274,4.8574,0.2151,0.0059,0.1890,4315.3953,0.2047,0.1755,0.2689,0.1854,88228.9871,0.0755,3.5443,0.0772,0.0859,0.1149,0.0634,0.6746,0.4808,0.5529,0.4033,0.1413,0.0401,0.1618,0.2406,1.5212
MLP (agentic),3477.9205,0.4372,0.1400,520.4658,30566.9305,0.7461,0.0128,0.1649,712.4840,0.1424,0.3349,0.2649,7.3969,0.1300,0.1912,0.0167,0.8323,0.9718,1.0652,0.2389,4.6435,0.2148,0.0051,0.1787,4343.1520,0.2048,0.1712,0.2612,0.1910,86731.7915,0.0741,3.6026,0.0691,0.0824,0.0933,0.1015,0.6524,0.4812,0.5534,0.3952,0.1416,0.0386,0.1621,0.2390,1.5136
LightGBM,3451.5506,0.4332,0.1397,539.3348,31361.6368,0.7416,0.0071,0.1725,718.2746,0.1207,0.3378,0.2582,7.5184,0.1305,0.1932,0.0146,0.8326,0.9656,1.6071,0.2286,4.7099,0.2122,0.0059,0.1969,4271.0724,0.2094,0.1560,0.2467,0.1914,91025.7222,0.0752,3.5027,0.0420,0.0943,0.1027,0.0572,0.6151,0.4825,0.5469,0.4262,0.1321,0.0394,0.1617,0.2488,1.4619
LightGBM (agentic),3450.0840,0.4334,0.1408,534.2042,31338.6313,0.7416,0.0070,0.1619,719.6040,0.1310,0.3339,0.2625,7.2845,0.1319,0.1947,0.0156,0.8274,0.9625,1.3727,0.2322,4.6605,0.2109,0.0056,0.2008,4316.3859,0.2042,0.1524,0.2567,0.1874,90187.7524,0.0733,3.2991,0.0381,0.0921,0.0903,0.0649,0.6086,0.4824,0.5465,0.4230,0.1324,0.0389,0.1617,0.2538,1.4657
TabM,3455.1369,0.4302,0.1379,520.8429,30441.0468,0.7414,0.0067,0.1693,727.6719,0.1274,0.3303,0.2596,7.7018,0.1310,0.1883,0.0149,0.8474,0.9346,1.0746,0.2272,5.0098,0.2077,0.0048,0.2052,4111.7466,0.2038,0.1477,0.2672,0.1867,82867.5463,0.0722,3.3609,0.0412,0.0697,0.0958,0.0588,0.6378,0.4803,0.5501,0.4120,0.1356,0.0375,0.1610,0.2365,1.4669
TabM (agentic),3448.3530,0.4136,0.1386,518.3467,29945.0857,0.7429,0.0057,0.1699,719.0332,0.1380,0.3279,0.2603,7.3772,0.1306,0.1882,0.0154,0.8270,0.9414,1.1152,0.2318,4.3319,0.2142,0.0046,0.1691,4194.1777,0.2025,0.1468,0.2668,0.1846,83487.8701,0.0731,3.3444,0.0763,0.0731,0.1015,0.0521,0.6200,0.4802,0.5495,0.3990,0.1365,0.0370,0.1610,0.2386,1.4654
TabICL,3536.8469,0.3978,0.1359,500.3418,27962.4279,0.7631,

## Ensemble tables (greedy Caruana)

Same layout for the ensembles. Values are the ensemble test metric (no std). Only the
MLP, TabM and RealMLP families have ensembles.

In [5]:
main_table(ens_frame, ENS_METHODS, with_std=False)

dataset,black-friday,california,churn,diamond,house,microsoft,tabarena/APSFailure,tabarena/Amazon_employee_access,tabarena/Another-Dataset-on-used-Fiat-500,tabarena/Bioresponse,tabarena/Diabetes130US,tabarena/E-CommereShippingData,tabarena/Food_Delivery_Time,tabarena/GiveMeSomeCredit,tabarena/HR_Analytics_Job_Change_of_Data_Scientists,tabarena/NATICUSdroid,tabarena/QSAR-TID-11,tabarena/QSAR_fish_toxicity,tabarena/airfoil_self_noise,tabarena/bank-marketing,tabarena/concrete_compressive_strength,tabarena/credit_card_clients_default,tabarena/customer_satisfaction_in_airline,tabarena/diabetes,tabarena/healthcare_insurance_expenses,tabarena/heloc,tabarena/in_vehicle_coupon_recommendation,tabarena/jm1,tabarena/kddcup09_appetency,tabarena/miami_housing,tabarena/online_shoppers_intention,tabarena/physiochemical_protein,tabarena/polish_companies_bankruptcy,tabarena/qsar-biodeg,tabarena/splice,tabarena/taiwanese_bankruptcy_prediction,tabarena/wine_quality,tabred/cooking-time,tabred/delivery-eta,tabred/ecom-offers,tabred/homecredit-default,tabred/homesite-insurance,tabred/maps-routing,tabred/sberbank-housing,tabred/weather
metric,rmse,rmse,accuracy,rmse,rmse,rmse,roc_auc,roc_auc,rmse,roc_auc,roc_auc,roc_auc,rmse,roc_auc,roc_auc,roc_auc,rmse,rmse,rmse,roc_auc,rmse,roc_auc,roc_auc,roc_auc,rmse,roc_auc,roc_auc,roc_auc,roc_auc,rmse,roc_auc,rmse,roc_auc,roc_auc,log_loss,roc_auc,rmse,rmse,rmse,roc_auc,roc_auc,roc_auc,rmse,rmse,rmse
method,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
RealMLP,3445.8427,0.4054,0.8620,515.3554,30202.0683,0.7441,0.9925,0.8693,698.7137,0.8784,0.6733,0.7413,7.3778,0.8693,0.8113,0.9856,0.8284,0.9099,1.0711,0.7721,4.6129,0.7878,0.9951,0.8436,4156.0941,0.7998,0.8493,0.7645,0.8129,83396.5956,0.9279,3.1742,0.9690,0.9258,0.0956,0.9487,0.5969,0.4802,0.5465,0.5945,0.8634,0.9639,0.1610,0.2394,1.4361
RealMLP (agentic),3443.1805,0.3922,0.8610,515.8644,29877.4654,0.7425,0.9940,0.8549,720.8662,0.8757,0.6764,0.7425,7.5389,0.8693,0.8122,0.9860,0.8460,0.9139,1.0546,0.7733,4.4865,0.7908,0.9952,0.8281,4122.3904,0.7989,0.8517,0.7653,0.8202,80781.5013,0.9297,3.1772,0.9637,0.9315,0.0896,0.9426,0.5996,0.4805,0.5454,0.5932,0.8647,0.9642,0.1606,0.2267,1.4328
MLP,3460.8917,0.4417,0.8610,516.1831,30591.0276,0.7438,0.9939,0.8479,727.4892,0.8631,0.6652,0.7396,7.6165,0.8707,0.8112,0.9847,0.8591,0.9459,1.0921,0.7749,4.5909,0.7888,0.9946,0.8255,4242.8327,0.7957,0.8351,0.7390,0.8160,85496.1145,0.9268,3.3828,0.9370,0.9255,0.1169,0.9364,0.6437,0.4800,0.5508,0.5963,0.8619,0.9602,0.1611,0.2385,1.4923
MLP (agentic),3459.5230,0.4344,0.8630,510.6980,30085.8241,0.7446,0.9953,0.8570,713.9357,0.8701,0.6748,0.7370,7.4622,0.8702,0.8122,0.9857,0.8354,0.9204,1.0495,0.7726,4.1806,0.7892,0.9946,0.8353,4273.4431,0.7983,0.8431,0.7422,0.8095,83634.3212,0.9211,3.3040,0.9661,0.9318,0.0958,0.9433,0.6124,0.4803,0.5502,0.5994,0.8641,0.9599,0.1611,0.2476,1.4863
LightGBM,3453.7888,0.4272,0.8610,546.9685,31223.5738,0.7422,0.9882,0.8332,722.3254,0.8831,0.6674,0.7373,7.5217,0.8691,0.8077,0.9844,0.8366,0.9621,1.6305,0.7650,4.6905,0.7876,0.9942,0.8199,4289.9990,0.7947,0.8402,0.7587,0.7951,90131.0845,0.9243,3.7260,0.9561,0.9190,0.0986,0.9448,0.6223,0.4819,0.5464,0.5830,0.8679,0.9605,0.1616,0.2509,1.4592
LightGBM (agentic),3454.0026,0.4316,0.8625,528.8580,31270.2458,0.7435,0.9943,0.8558,715.0800,0.8849,0.6733,0.7415,7.3161,0.8663,0.8097,0.9846,0.8429,0.9251,1.6208,0.7632,4.4771,0.7864,0.9943,0.7923,4342.0762,0.7943,0.8393,0.7615,0.7152,90016.9898,0.9237,3.3345,0.9626,0.9053,0.0880,0.9390,0.6205,0.4826,0.5511,0.5848,0.8561,0.9602,0.1618,0.2778,1.4871
TabM,3456.4763,0.4157,0.8575,517.5136,30332.4710,0.7422,0.9940,0.8375,721.8324,0.8763,0.6730,0.7368,7.6896,0.8706,0.8129,0.9853,0.8413,0.9357,1.0417,0.7741,4.4148,0.7908,0.9952,0.7582,4115.6620,0.7966,0.8554,0.7431,0.8148,85352.1102,0.9287,3.3422,0.9621,0.9299,0.1028,0.9451,0.6152,0.4799,0.5476,0.5857,0.8647,0.9623,0.1610,0.2339,1.4778
TabM (agentic),3459.9374,0.4079,0.8580,515.4094,29927.2148,0.7426,0.9954,0.8465,727.3506,0.8795,0.676

### Ensemble metric-error view (lower is better everywhere)

In [6]:
(ens_frame.pivot_table(index="method_key", columns="dataset", values="error", aggfunc="first")
 .reindex(index=ENS_METHODS, columns=[d for d in DATASETS if d in set(ens_frame["dataset"])])
 .rename(index=METHOD_DISPLAY)
 .round(4))

dataset,black-friday,california,churn,diamond,house,microsoft,tabarena/APSFailure,tabarena/Amazon_employee_access,tabarena/Another-Dataset-on-used-Fiat-500,tabarena/Bioresponse,tabarena/Diabetes130US,tabarena/E-CommereShippingData,tabarena/Food_Delivery_Time,tabarena/GiveMeSomeCredit,tabarena/HR_Analytics_Job_Change_of_Data_Scientists,tabarena/NATICUSdroid,tabarena/QSAR-TID-11,tabarena/QSAR_fish_toxicity,tabarena/airfoil_self_noise,tabarena/bank-marketing,tabarena/concrete_compressive_strength,tabarena/credit_card_clients_default,tabarena/customer_satisfaction_in_airline,tabarena/diabetes,tabarena/healthcare_insurance_expenses,tabarena/heloc,tabarena/in_vehicle_coupon_recommendation,tabarena/jm1,tabarena/kddcup09_appetency,tabarena/miami_housing,tabarena/online_shoppers_intention,tabarena/physiochemical_protein,tabarena/polish_companies_bankruptcy,tabarena/qsar-biodeg,tabarena/splice,tabarena/taiwanese_bankruptcy_prediction,tabarena/wine_quality,tabred/cooking-time,tabred/delivery-eta,tabred/ecom-offers,tabred/homecredit-default,tabred/homesite-insurance,tabred/maps-routing,tabred/sberbank-housing,tabred/weather
method_key,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
RealMLP,3445.8427,0.4054,0.1380,515.3554,30202.0683,0.7441,0.0075,0.1307,698.7137,0.1216,0.3267,0.2587,7.3778,0.1307,0.1887,0.0144,0.8284,0.9099,1.0711,0.2279,4.6129,0.2122,0.0049,0.1564,4156.0941,0.2002,0.1507,0.2355,0.1871,83396.5956,0.0721,3.1742,0.0310,0.0742,0.0956,0.0513,0.5969,0.4802,0.5465,0.4055,0.1366,0.0361,0.1610,0.2394,1.4361
RealMLP (agentic),3443.1805,0.3922,0.1390,515.8644,29877.4654,0.7425,0.0060,0.1451,720.8662,0.1243,0.3236,0.2575,7.5389,0.1307,0.1878,0.0140,0.8460,0.9139,1.0546,0.2267,4.4865,0.2092,0.0048,0.1719,4122.3904,0.2011,0.1483,0.2347,0.1798,80781.5013,0.0703,3.1772,0.0363,0.0685,0.0896,0.0574,0.5996,0.4805,0.5454,0.4068,0.1353,0.0358,0.1606,0.2267,1.4328
MLP,3460.8917,0.4417,0.1390,516.1831,30591.0276,0.7438,0.0061,0.1521,727.4892,0.1369,0.3348,0.2604,7.6165,0.1293,0.1888,0.0153,0.8591,0.9459,1.0921,0.2251,4.5909,0.2112,0.0054,0.1745,4242.8327,0.2043,0.1649,0.2610,0.1840,85496.1145,0.0732,3.3828,0.0630,0.0745,0.1169,0.0636,0.6437,0.4800,0.5508,0.4037,0.1381,0.0398,0.1611,0.2385,1.4923
MLP (agentic),3459.5230,0.4344,0.1370,510.6980,30085.8241,0.7446,0.0047,0.1430,713.9357,0.1299,0.3252,0.2630,7.4622,0.1298,0.1878,0.0143,0.8354,0.9204,1.0495,0.2274,4.1806,0.2108,0.0054,0.1647,4273.4431,0.2017,0.1569,0.2578,0.1905,83634.3212,0.0789,3.3040,0.0339,0.0682,0.0958,0.0567,0.6124,0.4803,0.5502,0.4006,0.1359,0.0401,0.1611,0.2476,1.4863
LightGBM,3453.7888,0.4272,0.1390,546.9685,31223.5738,0.7422,0.0118,0.1668,722.3254,0.1169,0.3326,0.2627,7.5217,0.1309,0.1923,0.0156,0.8366,0.9621,1.6305,0.2350,4.6905,0.2124,0.0058,0.1801,4289.9990,0.2053,0.1598,0.2413,0.2049,90131.0845,0.0757,3.7260,0.0439,0.0810,0.0986,0.0552,0.6223,0.4819,0.5464,0.4170,0.1321,0.0395,0.1616,0.2509,1.4592
LightGBM (agentic),3454.0026,0.4316,0.1375,528.8580,31270.2458,0.7435,0.0057,0.1442,715.0800,0.1151,0.3267,0.2585,7.3161,0.1337,0.1903,0.0154,0.8429,0.9251,1.6208,0.2368,4.4771,0.2136,0.0057,0.2077,4342.0762,0.2057,0.1607,0.2385,0.2848,90016.9898,0.0763,3.3345,0.0374,0.0947,0.0880,0.0610,0.6205,0.4826,0.5511,0.4152,0.1439,0.0398,0.1618,0.2778,1.4871
TabM,3456.4763,0.4157,0.1425,517.5136,30332.4710,0.7422,0.0060,0.1625,721.8324,0.1237,0.3270,0.2632,7.6896,0.1294,0.1871,0.0147,0.8413,0.9357,1.0417,0.2259,4.4148,0.2092,0.0048,0.2418,4115.6620,0.2034,0.1446,0.2569,0.1852,85352.1102,0.0713,3.3422,0.0379,0.0701,0.1028,0.0549,0.6152,0.4799,0.5476,0.4143,0.1353,0.0377,0.1610,0.2339,1.4778
TabM (agentic),3459.9374,0.4079,0.1420,515.4094,29927.2148,0.7426,0.0046,0.1535,727.3506,0.1205,0.3236,0.2602,7.4060,0.1288,0.1893,0.0135,0.8270,0.9389,0.9682,0.2275,4.4071,0.2073,0.0047,0.1752,4223.9061,0.2018,0.1488,0.2484,0.1834,81855.1413,0.0723,3.2783,0.0474,0.0628,0.0859,0.0570,0.6011,0.4799,0.5465,0.4007,0.1348,0.0429,0.1610,0.2420,1.4610


## Win rates over the baseline HP set

For each model family the agentic method is the *candidate* and the base method is the
*baseline HP set*. A win is a strictly lower test error on a dataset where both were run.
`improvement_pct` is the relative error reduction of agentic vs. baseline.

In [7]:
single_details = comparison_details(error_pivot(single_frame))

print("Single models -- by model family:")
display(summarize(single_details, ["family", "candidate", "baseline"]))

print("Single models -- by model family and task type:")
display(summarize(single_details, ["family", "task_type"]))

Single models -- by model family:


,family,candidate,baseline,datasets,wins,win_rate_pct,mean_improvement_pct,median_improvement_pct
0,LightGBM,LightGBM (agentic),LightGBM,45,27,60.0,0.73,0.32
1,MLP,MLP (agentic),MLP,45,27,60.0,-0.83,1.00
2,RealMLP,RealMLP (agentic),RealMLP,45,19,42.2,-0.24,-0.22
3,TabICL,TabICL (agentic),TabICL,45,29,64.4,1.52,0.20
4,TabM,TabM (agentic),TabM,45,26,57.8,-0.85,0.12


Single models -- by model family and task type:


,family,task_type,datasets,wins,win_rate_pct,mean_improvement_pct,median_improvement_pct
0,LightGBM,clf,25,14,56.0,0.32,0.76
1,LightGBM,reg,20,13,65.0,1.25,0.07
2,MLP,clf,25,13,52.0,-3.10,0.16
3,MLP,reg,20,14,70.0,2.02,1.19
4,RealMLP,clf,25,8,32.0,-0.54,-0.83
5,RealMLP,reg,20,11,55.0,0.13,0.14
6,TabICL,clf,25,16,64.0,2.23,0.32
7,TabICL,reg,20,13,65.0,0.63,0.19
8,TabM,clf,25,13,52.0,-2.43,0.01
9,TabM,reg,20,13,65.0,1.13,0.16


### Ensemble win rates over the baseline HP set

In [8]:
ens_details = comparison_details(error_pivot(ens_frame))

print("Ensembles -- by model family:")
display(summarize(ens_details, ["family", "candidate", "baseline"]))

print("Ensembles -- by model family and task type:")
display(summarize(ens_details, ["family", "task_type"]))

Ensembles -- by model family:


,family,candidate,baseline,datasets,wins,win_rate_pct,mean_improvement_pct,median_improvement_pct
0,LightGBM,LightGBM (agentic),LightGBM,45,22,48.9,0.35,-0.01
1,MLP,MLP (agentic),MLP,45,35,77.8,3.62,1.57
2,RealMLP,RealMLP (agentic),RealMLP,45,28,62.2,0.21,0.34
3,TabM,TabM (agentic),TabM,45,31,68.9,1.74,0.87


Ensembles -- by model family and task type:


,family,task_type,datasets,wins,win_rate_pct,mean_improvement_pct,median_improvement_pct
0,LightGBM,clf,25,13,52.0,0.23,0.43
1,LightGBM,reg,20,9,45.0,0.50,-0.05
2,MLP,clf,25,19,76.0,5.24,1.44
3,MLP,reg,20,16,80.0,1.59,1.65
4,RealMLP,clf,25,16,64.0,-0.03,0.50
5,RealMLP,reg,20,12,60.0,0.51,0.21
6,TabM,clf,25,18,72.0,2.40,0.94
7,TabM,reg,20,13,65.0,0.92,0.30
